#### Excel table 作り Logistic Regressionの結果

In [1]:
import numpy as np
import pandas as pd


print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

NumPy version: 1.26.4
Pandas version: 2.3.3


In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# 1. データの準備 (High, Middle, Low の3つのDataFrameがあると仮定)
# ここではダミーデータを作成します
np.random.seed(42)
def create_dummy_data(n):
    return pd.DataFrame({
        'SAS': np.random.randint(0, 2, n), # 目的変数 (0 or 1)
        'METs_cat': np.random.choice(['Q1', 'Q2', 'Q3', 'Q4'], n), # 説明変数
        'Alcohol': np.random.choice(['Yes', 'No'], n),
        'Age': np.random.randint(40, 70, n),
        'Sex': np.random.randint(0, 2, n)
    })

High = create_dummy_data(300)
Middle = create_dummy_data(300)
Low = create_dummy_data(300)

In [3]:
High

,SAS,METs_cat,Alcohol,Age,Sex
0,0,Q1,Yes,68,0
1,1,Q1,No,51,1
2,0,Q3,No,61,0
3,0,Q2,No,46,0
4,0,Q2,Yes,66,0
...,...,...,...,...,...
295,1,Q4,No,56,1
296,1,Q1,No,53,0
297,0,Q3,No,65,0
298,0,Q4,Yes,52,1


In [4]:
formula = "SAS ~ C(METs_cat, Treatment(reference='Q1')) + Age + Sex"
model = smf.logit(formula, data=High).fit(disp=0)
    
    # オッズ比と信頼区間を取得
odds_ratios = np.exp(model.params)
conf = np.exp(model.conf_int())
conf.columns = ['Lower', 'Upper']
p_values = model.pvalues

In [5]:
model.params.index[3]

"C(METs_cat, Treatment(reference='Q1'))[T.Q4]"

In [6]:
model.params.index[2]
label = model.params.index[2].split('[T.')[1].replace(']', '')
label

'Q3'

In [7]:
l = []
for var in model.params.index[1:]: # interceptを除く
    variable = var.split(',')[0].replace('C(','')
    l.append(variable)

In [8]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                    SAS   No. Observations:                  300
Model:                          Logit   Df Residuals:                      294
Method:                           MLE   Df Model:                            5
Date:                Fri, 02 Jan 2026   Pseudo R-squ.:                0.002164
Time:                        18:42:48   Log-Likelihood:                -207.49
converged:                       True   LL-Null:                       -207.94
Covariance Type:            nonrobust   LLR p-value:                    0.9702
================================================================================================================
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
Intercept                                       -0.2897      0.759     -0.382      0.703      -1.778       1.198
C(METs_cat, Treatment(reference='Q1'))[T.Q2]    -0.0878      0.341     -0.258      0.797      -0.755       0.580
C(METs_cat, Treatment(reference='Q1'))[T.Q3]     0.1893      0.345      0.549      0.583      -0.486       0.865
C(METs_cat, Treatment(reference='Q1'))[T.Q4]     0.0528      0.320      0.165      0.869      -0.575       0.681
Age                                              0.0046      0.013      0.349      0.727      -0.021       0.030
Sex                                             -0.0279      0.232     -0.120      0.904      -0.482       0.427
================================================================================================================
"""

In [9]:
r = model.params.index.unique()
r

Index(['Intercept', 'C(METs_cat, Treatment(reference='Q1'))[T.Q2]',
       'C(METs_cat, Treatment(reference='Q1'))[T.Q3]',
       'C(METs_cat, Treatment(reference='Q1'))[T.Q4]', 'Age', 'Sex'],
      dtype='object')

In [10]:
odds_ratios = np.exp(model.params)
odds_ratios

Intercept                                       0.748505
C(METs_cat, Treatment(reference='Q1'))[T.Q2]    0.915930
C(METs_cat, Treatment(reference='Q1'))[T.Q3]    1.208385
C(METs_cat, Treatment(reference='Q1'))[T.Q4]    1.054184
Age                                             1.004583
Sex                                             0.972489
dtype: float64

In [11]:
import re
pattern = r"C\((?P<var>.+?),\s*Treatment\(reference='(?P<ref>.+?)'\)\)\[T\.(?P<level>.+?)\]"

table_rows = []
processed_vars = set()

for idx in r:
    match = re.search(pattern, idx)
    if match:
        var_name = match.group('var')  # METs_cat
        ref_level = match.group('ref') # Q1
        level = match.group('level')   # Q2, Q3...
        
        # 新しい変数が見つかった場合、ヘッダーとリファレンスを最初に追加
        if var_name not in processed_vars:
            table_rows.append([var_name, ''])     # [METs_cat, '']       
            table_rows.append([ref_level, 'ref']) # [Q1, ref]
            processed_vars.add(var_name)
        
        # 現在のカテゴリを追加（値は仮に '...' としています）
        table_rows.append([level, '...'])

# 3. DataFrame化
df_result = pd.DataFrame(table_rows, columns=['Variable', 'Value'])

print(df_result)

   Variable Value
0  METs_cat      
1        Q1   ref
2        Q2   ...
3        Q3   ...
4        Q4   ...


In [12]:
import pandas as pd
import numpy as np
import re

# 1. 提供された結果データを辞書形式で定義 (実際の statsmodels の結果 df を想定)
# ユーザーが提示した数値を反映させています
results_data = {
    "C(METs_cat, Treatment(reference='Q1'))[T.Q2]": [-0.1305, 0.704, -0.803, 0.542],
    "C(METs_cat, Treatment(reference='Q1'))[T.Q3]": [ 0.2129, 0.536, -0.461, 0.886],
    "C(METs_cat, Treatment(reference='Q1'))[T.Q4]": [ 0.0521, 0.871, -0.577, 0.681],
    "Age": [-0.0085, 0.521, -0.034, 0.017],
    "Sex": [ 0.2621, 0.263, -0.197, 0.721]
}

# 解析用の正規表現
pattern = r"C\((?P<var>.+?),\s*Treatment\(reference='(?P<ref>.+?)'\)\)\[T\.(?P<level>.+?)\]"

table_rows = []
processed_vars = set()

# 2. データのループ処理
for idx, values in results_data.items():
    match = re.search(pattern, idx)
    
    if match:
        var_name = match.group('var')
        ref_level = match.group('ref')
        level = match.group('level')
        
        coef, p_val, lower, upper = values
        
        # 新しい変数の場合：ヘッダー、空行、リファレンス行を追加
        if var_name not in processed_vars:
            table_rows.append([var_name, '', '', ''])   # 変数名ラベル
            # table_rows.append(['', '', '', ''])         # 空行
            table_rows.append([ref_level, '(ref)', '-', '-']) # リファレンス行
            processed_vars.add(var_name)
        
        # 指数変換 (OR と CI)
        or_val = np.exp(coef)
        ci_low = np.exp(lower)
        ci_high = np.exp(upper)
        
        # 行の追加 (数値を綺麗にフォーマット)
        table_rows.append([
            level,
            f"{or_val:.2f}",
            f"{ci_low:.2f} - {ci_high:.2f}",
            f"{p_val:.3f}"
        ])

# 3. DataFrameの作成
final_df = pd.DataFrame(table_rows, columns=['Variable', 'OR', '95% CI', 'P-value'])

# 表示
print(final_df.to_string(index=False))

Variable    OR      95% CI P-value
METs_cat                          
      Q1 (ref)           -       -
      Q2  0.88 0.45 - 1.72   0.704
      Q3  1.24 0.63 - 2.43   0.536
      Q4  1.05 0.56 - 1.98   0.871


In [13]:
final_df

,Variable,OR,95% CI,P-value
0,METs_cat,,,
1,Q1,(ref),-,-
2,Q2,0.88,0.45 - 1.72,0.704
3,Q3,1.24,0.63 - 2.43,0.536
4,Q4,1.05,0.56 - 1.98,0.871


### アルゴリズム探索：

In [14]:
# Logistic Regression Result df 作成

import statsmodels.stats.multitest as smm
import statsmodels.api as sm

class LogiticRegression:
    def __init__(self, df): 

        if df.empty:
            raise ValueError("入力されたDataFrameが空です。")
        self.df = df

    """
    Categoryデータに関しては、変数をLevel0~4のようにRefが0となるような順番にコーディングすること: 
    """

    def logistic_regression_creator(self,continuous_col,categorical_col,target_col):
        
        sample_df = self.df[target_col+continuous_col+categorical_col]
        
        features = []
        features.extend(continuous_col)

        for col in categorical_col:
            features.append(f'C({col}, Treatment(reference=0))')

        formula = f'{target_col} ~ {" + ".join(features)}'
        print(f"Generated Formula: {formula}")

        model = sm.logit(formula = formula, data = sample_df)
        result = model.fit()

        result = model.fit()
        ci_exp = np.exp(result.conf_int())
        
        # ここからOrganizeしていく
        df_result = pd.DataFrame({
        'ORs': np.exp(result.params),
        'Coef (係数)': result.params,
        'P-value': result.pvalues,
        'CI_Upper':ci_exp[1],
        'CI_Lower':ci_exp[0],
        'Std.Err': result.bse
    
        })


        return ci_exp,df_result 


    """
    Formulaを事前に指定していれるVersion 
    """


    def logistic_regression_formula(self,formula):


        model = sm.Logit.from_formula(formula, data=self.df)
        result = model.fit()
        ci_exp = np.exp(result.conf_int())
        
        # ここからOrganizeしていく
        df_result = pd.DataFrame({
        'ORs': np.exp(result.params),
        'Coef': result.params,
        'P-value': result.pvalues,
        'CI_Upper':ci_exp[1],
        'CI_Lower':ci_exp[0],
        'Std.Err': result.bse
    
        })
        return df_result
    
    """
    Logistic RegressiongあとのDataFrame作成
    """


    def result_show(self,result):


        res = []
        for i in range(len(result)):
            r = result.iloc[i:i+1]
            index_name = r.index[0]

            if index_name == 'Intercept':
                continue

            find = re.findall(pattern, index_name)
            # res.append({'Variable': find[0][0],
            #             'Category': find[0][2],
            #             'Treatment': find[0][1],
            #             'ORs': r['ORs'].values[0],
            #             'CI_Lower': r['CI_Lower'].values[0],
            #             'CI_Upper': r['CI_Upper'].values[0],
            #             'P-value': r['P-value'].values[0]
            #             })
            ors_val = r['ORs'].values[0]
            lower_val = r['CI_Lower'].values[0]
            upper_val = r['CI_Upper'].values[0]
            p_val = r['P-value'].values[0]

            # P値の整形ロジック
            if p_val < 0.001:
                p_str = '< 0.001'
            else:
                p_str = f'{p_val:.3f}'

            
            res.append({
            'Variable': find[0][0],
            'Category': find[0][2],
            'Treatment': find[0][1],
            
            # :.3f をつけることで小数点3桁に固定
            # 形: "1.234 [0.900-1.500]"
            'ORs': f'{ors_val:.3f} [{lower_val:.3f}-{upper_val:.3f}]',
            'Odds': ors_val,
            'CI_Lower': lower_val,
            'CI_Upper': upper_val,
            # P値も同様に3桁にする場合
            'P-value': p_str
            })

        data = pd.DataFrame(res)

        """
        Bonferroni 
        """
        data['P-value']=pd.to_numeric(data['P-value'], errors='coerce')
        p_values = data['P-value']


        corrected_result = smm.multipletests(p_values, alpha=0.05, method='bonferroni')
        pval = corrected_result[1]
        data['Bonferroni_P-value'] = pval
        

        """
        Holm 
        """
        corrected_result = smm.multipletests(p_values, alpha=0.05, method='bonferroni')
        pval = corrected_result[1]

        data['Holm_P-value']=pval

        return data

In [15]:
Logistic = LogiticRegression(High)
result = Logistic.logistic_regression_formula('SAS ~ C(METs_cat, Treatment(reference="Q1")) + C(Alcohol, Treatment(reference="No")) + Age + Sex')
result

Optimization terminated successfully.
         Current function value: 0.690946
         Iterations 4


,ORs,Coef,P-value,CI_Upper,CI_Lower,Std.Err
Intercept,0.672603,-0.396600,0.610285,3.091186,0.146350,0.778154
"C(METs_cat, Treatment(reference=""Q1""))[T.Q2]",0.920336,-0.083016,0.807580,1.795093,0.471852,0.340860
"C(METs_cat, Treatment(reference=""Q1""))[T.Q3]",1.194214,0.177489,0.607313,2.349955,0.606883,0.345367
"C(METs_cat, Treatment(reference=""Q1""))[T.Q4]",1.053541,0.052157,0.870799,1.975233,0.561933,0.320684
"C(Alcohol, Treatment(reference=""No""))[T.Yes]",1.162315,0.150413,0.523317,1.844697,0.732356,0.235669
Age,1.004924,0.004911,0.708408,1.031125,0.979388,0.013132
Sex,0.997500,-0.002503,0.991517,1.582388,0.628800,0.235432


In [16]:
result_data = dict(result)

In [17]:
result_data['ORs'].index

Index(['Intercept', 'C(METs_cat, Treatment(reference="Q1"))[T.Q2]',
       'C(METs_cat, Treatment(reference="Q1"))[T.Q3]',
       'C(METs_cat, Treatment(reference="Q1"))[T.Q4]',
       'C(Alcohol, Treatment(reference="No"))[T.Yes]', 'Age', 'Sex'],
      dtype='object')

In [18]:
result_data

{'ORs': Intercept                                       0.672603
 C(METs_cat, Treatment(reference="Q1"))[T.Q2]    0.920336
 C(METs_cat, Treatment(reference="Q1"))[T.Q3]    1.194214
 C(METs_cat, Treatment(reference="Q1"))[T.Q4]    1.053541
 C(Alcohol, Treatment(reference="No"))[T.Yes]    1.162315
 Age                                             1.004924
 Sex                                             0.997500
 Name: ORs, dtype: float64,
 'Coef': Intercept                                      -0.396600
 C(METs_cat, Treatment(reference="Q1"))[T.Q2]   -0.083016
 C(METs_cat, Treatment(reference="Q1"))[T.Q3]    0.177489
 C(METs_cat, Treatment(reference="Q1"))[T.Q4]    0.052157
 C(Alcohol, Treatment(reference="No"))[T.Yes]    0.150413
 Age                                             0.004911
 Sex                                            -0.002503
 Name: Coef, dtype: float64,
 'P-value': Intercept                                       0.610285
 C(METs_cat, Treatment(reference="Q1"))[T.Q2]  

#### Dict にしたLogisticRegressionResultに対して以下の処理を行うのが一番わかりやすいと思う

In [19]:
# 解析用の正規表現
pattern = r"C\((?P<var>.+?),\s*Treatment\(reference=['\"](?P<ref>.+?)['\"]\)\)\[T\.(?P<level>.+?)\]"

a = []
for idx in result_data['ORs'].index:
    match = re.search(pattern, idx)
    a.append({'index': idx, 'match': match})
    

In [20]:
a

[{'index': 'Intercept', 'match': None},
 {'index': 'C(METs_cat, Treatment(reference="Q1"))[T.Q2]',
  'match': <re.Match object; span=(0, 44), match='C(METs_cat, Treatment(reference="Q1"))[T.Q2]'>},
 {'index': 'C(METs_cat, Treatment(reference="Q1"))[T.Q3]',
  'match': <re.Match object; span=(0, 44), match='C(METs_cat, Treatment(reference="Q1"))[T.Q3]'>},
 {'index': 'C(METs_cat, Treatment(reference="Q1"))[T.Q4]',
  'match': <re.Match object; span=(0, 44), match='C(METs_cat, Treatment(reference="Q1"))[T.Q4]'>},
 {'index': 'C(Alcohol, Treatment(reference="No"))[T.Yes]',
  'match': <re.Match object; span=(0, 44), match='C(Alcohol, Treatment(reference="No"))[T.Yes]'>},
 {'index': 'Age', 'match': None},
 {'index': 'Sex', 'match': None}]

In [72]:
table_rows = []
processed_vars = set()


for idx in result_data['ORs'].index:
    match = re.search(pattern, idx)
    
    if match:
        var_name = match.group('var')
        ref_level = match.group('ref')
        level = match.group('level')
        
        # 変数名が初めて出た時にヘッダーとリファレンス行を追加
        if var_name not in processed_vars:
            table_rows.append([var_name, '', '', ''])   # 変数名
            #table_rows.append(['', '', '', ''])         # 空行
            table_rows.append([ref_level, '1.0 (ref)', '-', '-']) # リファレンス
            processed_vars.add(var_name)
        # 各数値を各 Series から取得してフォーマット
        or_val = result_data['ORs'][idx]
        p_val = result_data['P-value'][idx]
        low = result_data['CI_Lower'][idx]
        up = result_data['CI_Upper'][idx]
        
        table_rows.append([
            level,
            f"{or_val:.2f}",
            f"{low:.2f} - {up:.2f}",
            f"{p_val:.3f}"
        ])

    else:
        # Numerical values: 
        display_name = idx
        if display_name == 'Intercept':
            continue  # skipping Intercept 
        else:
    
           
            or_val = result_data['ORs'][idx]
            p_val = result_data['P-value'][idx]
            low = result_data['CI_Lower'][idx]
            up = result_data['CI_Upper'][idx]
            
            table_rows.append([
                display_name.ljust(10),
                f"{or_val:.2f}",
                f"{low:.2f} - {up:.2f}",
                f"{p_val:.3f}"
            ])
    # df_final = pd.DataFrame(table_rows, columns=['Variable', 'OR', '95% CI', 'P-value']).style.set_properties(**{'text-align': 'left'})\
    #                       .set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    df_final = pd.DataFrame(table_rows, columns=['Variable', 'OR', '95% CI', 'P-value'])

df_final

,Variable,OR,95% CI,P-value
0,METs_cat,,,
1,Q1,1.0 (ref),-,-
2,Q2,0.92,0.47 - 1.80,0.808
3,Q3,1.19,0.61 - 2.35,0.607
4,Q4,1.05,0.56 - 1.98,0.871
5,Alcohol,,,
6,No,1.0 (ref),-,-
7,Yes,1.16,0.73 - 1.84,0.523
8,Age,1.00,0.98 - 1.03,0.708
9,Sex,1.00,0.63 - 1.58,0.992


In [73]:
def bonferroni_correction(data):
    data['P-value']=pd.to_numeric(data['P-value'], errors='coerce')
    p_values = data['P-value']


    corrected_result = smm.multipletests(p_values, alpha=0.05, method='bonferroni')
    pval = corrected_result[1]
    data['Bonferroni_P-value'] = pval
    data['P-value'] = data['P-value'].replace(np.nan, '-')
    data['Bonferroni_P-value'] = data['Bonferroni_P-value'].replace(np.nan, '-')

    return data
        

In [81]:
corrected = bonferroni_correction(df_final)#.set_index('Variable')

In [82]:
corrected

,Variable,OR,95% CI,P-value,Bonferroni_P-value
0,METs_cat,,,-,-
1,Q1,1.0 (ref),-,-,-
2,Q2,0.92,0.47 - 1.80,0.808,1.0
3,Q3,1.19,0.61 - 2.35,0.607,1.0
4,Q4,1.05,0.56 - 1.98,0.871,1.0
5,Alcohol,,,-,-
6,No,1.0 (ref),-,-,-
7,Yes,1.16,0.73 - 1.84,0.523,1.0
8,Age,1.00,0.98 - 1.03,0.708,1.0
9,Sex,1.00,0.63 - 1.58,0.992,1.0


In [83]:
d = corrected.set_index(['Variable'])

In [84]:
d

,OR,95% CI,P-value,Bonferroni_P-value
Variable,,,,
METs_cat,,,-,-
Q1,1.0 (ref),-,-,-
Q2,0.92,0.47 - 1.80,0.808,1.0
Q3,1.19,0.61 - 2.35,0.607,1.0
Q4,1.05,0.56 - 1.98,0.871,1.0
Alcohol,,,-,-
No,1.0 (ref),-,-,-
Yes,1.16,0.73 - 1.84,0.523,1.0
Age,1.00,0.98 - 1.03,0.708,1.0


In [85]:
corrected.columns

Index(['Variable', 'OR', '95% CI', 'P-value', 'Bonferroni_P-value'], dtype='object')

In [86]:

corrected.columns = pd.MultiIndex.from_product([['Model 1'], corrected.columns])

# 3. 表示
corrected #= corrected.set_index(['Variable'])

Model 1                                                   
     Variable         OR       95% CI P-value Bonferroni_P-value
0    METs_cat                               -                  -
1          Q1  1.0 (ref)            -       -                  -
2          Q2       0.92  0.47 - 1.80   0.808                1.0
3          Q3       1.19  0.61 - 2.35   0.607                1.0
4          Q4       1.05  0.56 - 1.98   0.871                1.0
5     Alcohol                               -                  -
6          No  1.0 (ref)            -       -                  -
7         Yes       1.16  0.73 - 1.84   0.523                1.0
8  Age              1.00  0.98 - 1.03   0.708                1.0
9  Sex              1.00  0.63 - 1.58   0.992                1.0

In [88]:
import pandas as pd

data_m2 = [
    ['Q1', '1.0 (ref)', '-', '-', '-'],
    ['Q2', '0.95', '0.50-1.80', '0.800', '1.000'],
    ['Q3', '1.10', '0.55-2.10', '0.400', '0.800'],
    ['Q4', '1.02', '0.50-2.05', '0.900', '1.000'],
    ['BMI', '1.05', '1.01-1.10', '0.020', '0.040'] # Model 2にしかない変数
]
cols_m2 = ['Variable', 'OR', '95% CI', 'P-value', 'Bonferroni_P-value']
df_model2 = pd.DataFrame(data_m2, columns=cols_m2).set_index('Variable')

df_model2.columns = pd.MultiIndex.from_product([['Model 2'], df_model2.columns])


# Combine models 
df_combined = pd.merge(df_final, df_model2, left_index=True, right_index=True, how='outer')
# df_combined = pd.concat([corrected, df_model2], axis=1)


# df_combined = df_combined.fillna('')

# df_combined

# Mergeするには、生データの時点でMergeしないと変数が混ざってしまう